### TODO:
1. Load multiple dataset for different buildings.
2. Get the climate info for the state.
3. preprocess the data.
4. Create a linear regression model.
5. Prepare some tests for the model.

In [ ]:
import polars as pl
import pyarrow
import pandas as pd

In [ ]:
from pathlib import Path
folder= Path().resolve().parent /'input'


### 1. Load multiple dataset for different buildings.

In [ ]:
# read the original data for three individual buildings
data=pl.scan_parquet(folder / 'load_profile_buildingID_*')

In [ ]:
# extract the buildings Ids
bldgs=data.select(pl.col('bldg_id').unique()).collect().to_series().to_list()

### 2. Get the climate info for the state.

In [152]:
# extract the climate zone details based on the builings and cast the building id type to match the one in the original data
MetaData=(
    pl.scan_parquet(folder / 'MetaData.parquet')
            .filter(
            pl.col('bldg_id').is_in(bldgs)).cast({"bldg_id": pl.UInt32}).select(
            pl.col('bldg_id', 'in.ashrae_iecc_climate_zone_2004_sub_cz_split'))
)
# MetaData.row(0, named=True) 
MetaData.collect()

bldg_id,in.ashrae_iecc_climate_zone_2004_sub_cz_split
u32,str
10,"""1A - FL"""
100016,"""2A - FL, GA, AL, MS"""
100024,"""1A - FL"""


In [159]:
%%time
# merge the climate zone changes into the original data
df=data.join(MetaData, on='bldg_id')

CPU times: user 62.4 ms, sys: 0 ns, total: 62.4 ms
Wall time: 28.3 ms


in.ashrae_iecc_climate_zone_2004_sub_cz_split
str
"""2A - FL, GA, AL, MS"""
"""1A - FL"""


There are only time climate zones for the data

----------------------------------------------

### Prepare cross-validation model


In [ ]:
df.collect_schema()

In [160]:
# defining the independant and dependant variables with encoding categorical variables and converting the datetime to a timestamp
x= df.select(pl.col("timestamp").dt.timestamp(), pl.col('out.electricity.total.energy_consumption..kwh').alias('Total Usage'),
             pl.col('in.ashrae_iecc_climate_zone_2004_sub_cz_split').cast(pl.Categorical).alias('Climate Zone')).collect()
y=df.select(pl.col("out.electricity.cooling.energy_consumption..kwh").alias('Cooling Consumption'),
           pl.col("out.electricity.heating.energy_consumption..kwh").alias('Heating Consumption'),
           pl.col("out.electricity.freezer.energy_consumption..kwh").alias('Freezing Consumption'),
           pl.col("out.electricity.refrigerator.energy_consumption..kwh").alias('Refrigerator Consumption'),
           pl.col("out.electricity.dishwasher.energy_consumption..kwh").alias('Dishwasher Consumption')).collect()
x.select(pl.col('Climate Zone').unique())

Climate Zone
cat
"""1A - FL"""
"""2A - FL, GA, AL, MS"""


In [139]:
def model(mod, x_train, x_test, y_train, y_test):
    ml=mod.fit(x_train, y_train)
    return ml.score(x_test, y_test)

In [140]:
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
skf= StratifiedKFold(n_splits=10)
kf=KFold(n_splits=10)
arr=[]
lf=pd.DataFrame()
for i, (train,test) in enumerate(kf.split(x,y)):
    x_train, x_test, y_train, y_test=x[train], x[test], y[train], y[test]
    score=model(LinearRegression(), x_train, x_test, y_train, y_test)
    frame=pd.DataFrame({
    "Split Number": [i],
    "Score": [score]
    })
    arr.append(frame)
lf=pd.concat(arr)
lf

ValueError: could not convert string to float: '1A - FL'